<a href="https://colab.research.google.com/github/SandiFerdiyansyah/Analisis-aplikasi-pemerintahan-Sentuh-Tanahku/blob/main/Sandi%20Ferdiyansyah_14022300003_aplikasi_pemerintahan_Sentuh_Tanahku_BPN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install google-play-scraper

Analisis aplikasi pemerintahan Sentuh Tanahku BPN

1.   List item
2.   List item



In [ ]:
from google_play_scraper import reviews, Sort
import csv

result, _ = reviews(
    'id.go.bpn.sentuh',
    lang='id',
    country='id',
    sort=Sort.NEWEST,
    count=1000,
    filter_score_with=None
)

filename = 'ulasan_google_play.csv'


with open(filename, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['userName', 'score', 'at', 'content'])
    writer.writeheader()
    for review in result:

        writer.writerow({
            'userName': review['userName'],
            'score': review['score'],
            'at': review['at'],
            'content': review['content']
        })

print(f"Berhasil menyimpan {len(result)} ulasan ke '{filename}'")

In [ ]:
pip install transformers torch pandas

Analisis Sentimen Menggunakan Indo-RoBERTa

In [ ]:
import pandas as pd
from transformers import pipeline

# Muat data dari file CSV yang sudah dibuat sebelumnya
df = pd.read_csv('/content/ulasan_google_play.csv')

# Inisialisasi pipeline sentiment analysis dengan model Indo-RoBERTa
# Model ini mengklasifikasikan teks ke dalam 3 label: Positive, Neutral, Negative
sentiment_pipeline = pipeline("sentiment-analysis", model="w11wo/indonesian-roberta-base-sentiment-classifier")

def get_sentiment(text):
    try:
        # Batasi teks agar tidak melebihi kapasitas model (biasanya 512 token)
        result = sentiment_pipeline(str(text)[:512])[0]
        return result['label'], result['score']
    except:
        return "Error", 0.0

# Jalankan analisis (ini mungkin memakan waktu beberapa menit tergantung jumlah data)
print("Sedang memproses sentimen...")
df[['label', 'confidence']] = df['content'].apply(lambda x: pd.Series(get_sentiment(x)))

# Tampilkan hasil
display(df.head())


In [ ]:
# Ringkasan Statistik Sentimen
print("Ringkasan Sentimen:")
print(df['label'].value_counts())

# Simpan hasil analisis ke CSV baru
df.to_csv('ulasan_google_play_analyzed.csv', index=False)
print("\nHasil analisis disimpan ke 'ulasan_google_play_analyzed.csv'")

Visualisasi Hasil Analisis Sentimen

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Mengatur gaya visualisasi
sns.set(style="whitegrid")

# Membuat figure dengan dua subplot
fig, ax = plt.subplots(1, 2, figsize=(15, 6))

# 1. Bar Chart: Distribusi Jumlah Sentimen
sns.countplot(x='label', data=df, hue='label', palette='viridis', ax=ax[0], legend=False)
ax[0].set_title('Distribusi Jumlah Sentimen')
ax[0].set_xlabel('Label Sentimen')
ax[0].set_ylabel('Jumlah Ulasan')

# 2. Pie Chart: Persentase Sentimen
sentiment_counts = df['label'].value_counts()
ax[1].pie(sentiment_counts, labels=sentiment_counts.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette('viridis', len(sentiment_counts)))
ax[1].set_title('Persentase Sentimen')

plt.tight_layout()

# Menyimpan grafik ke dalam file
plt.savefig('visualisasi_sentimen.png', dpi=300)
print("Grafik telah disimpan sebagai 'visualisasi_sentimen.png'")

plt.show()

### Daftar Ulasan Negatif Panjang (20+ Kata)

Berikut adalah ulasan negatif mendetail yang dapat digunakan untuk menganalisis kesalahan atau masalah pada aplikasi.

In [ ]:
# Menampilkan ulasan negatif yang memiliki 20 kata atau lebih secara utuh
long_negative_reviews = df[(df['label'] == 'negative') & (df['content'].str.split().str.len() >= 20)][['userName', 'content']]

pd.set_option('display.max_colwidth', None)

# Menyimpan hasil ulasan negatif panjang ke file CSV
long_negative_reviews.to_csv('ulasan_negatif_panjang.csv', index=False)

print(f"Menampilkan {len(long_negative_reviews)} ulasan negatif mendetail:")
print(f"Daftar ini juga telah disimpan ke dalam file 'ulasan_negatif_panjang.csv'")
display(long_negative_reviews)